In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 6a: Rekayasa fitur

Cerita lengkapnya ada di `README.md`. Di sini: hitung dulu, baru jalankan kode.

Empat gelas, dalam mililiter: 200, 50, 180, dan 40. Suhu baris pertama: dingin.

1. Berapa jumlah keempat angka itu?
2. Berapa hasil jumlah itu dibagi 4?
3. 200 mililiter sama dengan berapa liter? Bagi dengan 1000.
4. Kalau ada satu kolom untuk dingin dan satu kolom untuk panas, berisi 1 atau 0, kolom mana yang berisi 1 pada baris pertama?

Hitung dengan tangan, lalu jalankan sel dari atas ke bawah.


In [2]:
# Pustaka adalah paket kode siap pakai yang kita panggil.
# Array, istilah modul persiapan: kumpulan nilai sejenis dengan bentuk yang tetap, yang dihitung sekaligus.
# Satu ide: panggil NumPy, pustaka untuk hitungan pada array.
import numpy as np


In [3]:
# DataFrame, istilah modul persiapan: tabel di memori, setiap kolom punya nama, setiap baris satu catatan.
# Satu ide: panggil pandas, pustaka untuk tabel.
import pandas as pd


In [4]:
# Satu ide: simpan empat gelas. Angka di kolom gelas_ml, tulisan di kolom suhu.
catatan = pd.DataFrame(
    {
        "gelas_ml": [200, 50, 180, 40],
        "suhu": ["dingin", "panas", "dingin", "panas"],
    }
)
catatan


,gelas_ml,suhu
0,200,dingin
1,50,panas
2,180,dingin
3,40,panas


In [5]:
# Satu ide: jumlah seluruh kolom gelas.
jumlah = int(catatan["gelas_ml"].sum())
print("jumlah", jumlah)


jumlah 470


In [6]:
# Satu ide: rata-rata = jumlah dibagi banyaknya baris.
rata_rata = float(catatan["gelas_ml"].mean())
print("rata_rata", rata_rata)


rata_rata 117.5


In [7]:
# Fitur adalah kolom, atau hasil hitungan dari kolom, yang dipakai model untuk menebak.
# Satu ide: fitur baru, gelas dalam liter. Satu liter = 1000 mililiter.
catatan["gelas_liter"] = catatan["gelas_ml"] / 1000
print("liter", catatan["gelas_liter"].tolist())


liter [0.2, 0.05, 0.18, 0.04]


In [8]:
# Satu ide: mengganti tulisan dengan 0 dan 1 pada SATU kolom mengarang urutan.
# Di sini 0 bukan "lebih dingin" dan 1 bukan "dua kali dingin". Jangan dipakai bila tidak ada urutan.
urutan_palsu = catatan["suhu"].map({"dingin": 0, "panas": 1})
print("urutan_palsu", urutan_palsu.tolist())


urutan_palsu [0, 1, 0, 1]


## Data latih dan data uji

Istilah modul cara berpikir, satu arti: data dibagi dua. Data latih dipakai model untuk belajar. Data uji disisihkan dan tidak dilihat model saat belajar, dipakai untuk mengukur seberapa baik model pada data baru.

Indeks adalah nomor baris, mulai dari 0. Tiga baris pertama untuk belajar. Baris terakhir disisihkan.


In [9]:
# Satu ide: sisihkan baris terakhir sebagai data uji.
latih = catatan.iloc[:3]
uji = catatan.iloc[3:]
print("baris_latih", latih.index.tolist())
print("baris_uji", uji.index.tolist())


baris_latih [0, 1, 2]
baris_uji [3]


In [10]:
# Encoding mengubah kategori, yaitu teks berlabel terbatas, menjadi angka supaya model bisa menghitung.
# one-hot: satu kolom baru per kategori, 1 jika cocok dan 0 jika tidak.
# sparse_output=False meminta array biasa, yang menyimpan semua angka termasuk nol.
# Satu ide: panggil OneHotEncoder.
from sklearn.preprocessing import OneHotEncoder


In [11]:
# Satu ide: encoding belajar kategori dari data latih saja, lalu dipakai pada data uji.
pengkode = OneHotEncoder(sparse_output=False)
hasil_latih = pengkode.fit_transform(latih[["suhu"]])
hasil_uji = pengkode.transform(uji[["suhu"]])
print("nama_latih", pengkode.get_feature_names_out(["suhu"]).tolist())
print("kode_latih_pertama", hasil_latih[0].tolist())
print("kode_uji", hasil_uji[0].tolist())


nama_latih ['suhu_dingin', 'suhu_panas']
kode_latih_pertama [1.0, 0.0]
kode_uji [0.0, 1.0]


## Penskalaan

Penskalaan mengubah angka sebuah kolom supaya rentangnya sebanding dengan kolom lain.

Gambar dulu: gelas latih 200, 50, dan 180. Kolom one-hot hanya 0 atau 1. Yang ratusan akan menelan yang kecil bila keduanya dijumlah apa adanya.

Rumus, setelah gambar itu: tiap angka menjadi `(angka - rata-rata) / sebaran`.

Sebaran di sini adalah simpangan baku: seberapa jauh angka-angka tersebar dari rata-rata. StandardScaler membagi dengan banyaknya data, bukan dengan banyaknya data dikurangi satu. Rata-rata dan sebaran hanya dihitung dari data latih. Sel berikutnya mencetak angkanya.


In [12]:
# Satu ide: panggil StandardScaler, perkakas penskalaan itu.
from sklearn.preprocessing import StandardScaler


In [13]:
# fit artinya langkah ini belajar dari data latih.
# Satu ide: rata-rata dan sebaran hanya dari tiga gelas latih.
penimbang = StandardScaler()
penimbang.fit(latih[["gelas_ml"]])
print("rata_latih", float(penimbang.mean_[0]))
print("sebaran_latih", float(penimbang.scale_[0]))


rata_latih 143.33333333333334
sebaran_latih 66.4997911442


In [14]:
# transform artinya langkah yang sudah belajar mengubah data, tanpa belajar ulang.
# Satu ide: ubah gelas uji dengan timbangan yang sudah disetel dari data latih.
uji_skala = penimbang.transform(uji[["gelas_ml"]])
print("uji_skala", float(uji_skala[0, 0]))


uji_skala -1.5538895920629654


In [15]:
# Satu ide: cara yang salah. Rata-rata ini melihat baris uji juga.
# Data leakage, istilah modul evaluasi: informasi yang seharusnya tidak tersedia saat belajar ikut terpakai.
# Di sel ini kita belum mengukur skor. Yang terlihat adalah rata-ratanya berubah.
penimbang_salah = StandardScaler()
penimbang_salah.fit(catatan[["gelas_ml"]])
print("rata_semua", float(penimbang_salah.mean_[0]))
print("beda", float(penimbang.mean_[0]) != float(penimbang_salah.mean_[0]))


rata_semua 117.5
beda True


## Kasus nyata

Tabel di bawah dibuat di memori dengan pengacak yang dikunci `SEED`. Tidak ada unduhan. Ini bukan data asli perusahaan.

Aturan main yang kita tanam: jawaban `kembali` berisi ya bila `gelas_ml` di bawah 100. Kolom jawaban itu bukan fitur.


In [16]:
# Satu ide: buat dua belas baris. default_rng memakai SEED supaya tabel ini berulang.
rng = np.random.default_rng(SEED)
n = 12
gelas_ml = rng.integers(30, 220, size=n)
suhu = np.where(rng.random(n) < 0.5, "dingin", "panas")
kembali = np.where(gelas_ml < 100, "ya", "tidak")
kasus = pd.DataFrame(
    {
        "gelas_ml": gelas_ml,
        "suhu": suhu,
        "kembali": kembali,
    }
)
kasus


,gelas_ml,suhu,kembali
0,46,panas,ya
1,177,panas,tidak
2,154,dingin,tidak
3,113,dingin,tidak
4,112,dingin,tidak
5,193,panas,tidak
6,46,panas,ya
7,162,panas,tidak
8,68,dingin,ya
9,47,dingin,ya


In [17]:
# Kolom jawaban bukan fitur.
# Satu ide: sebutkan kolom yang dipakai dan kolom yang tidak.
print("kolom_fitur", ["gelas_ml", "suhu"])
print("kolom_jawaban", "kembali")


kolom_fitur ['gelas_ml', 'suhu']
kolom_jawaban kembali


In [18]:
# train_test_split membagi baris menjadi data latih dan data uji.
# Kocokan dikunci dengan random_state=SEED.
# Satu ide: panggil pembagi itu.
from sklearn.model_selection import train_test_split


In [19]:
# Satu ide: bagi tabel kasus. Jawaban tidak masuk ke X.
X = kasus[["gelas_ml", "suhu"]]
y = kasus["kembali"]
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
print("n_latih", len(X_latih))
print("n_uji", len(X_uji))


n_latih 9
n_uji 3


In [20]:
# Satu ide: penskalaan kasus ini juga hanya belajar dari data latih.
penimbang_kasus = StandardScaler()
penimbang_kasus.fit(X_latih[["gelas_ml"]])
print("kasus_rata_latih", float(penimbang_kasus.mean_[0]))
print("kasus_rata_semua", float(kasus["gelas_ml"].mean()))
print(
    "kasus_beda",
    float(penimbang_kasus.mean_[0]) != float(kasus["gelas_ml"].mean()),
)


kasus_rata_latih 137.77777777777777
kasus_rata_semua 121.91666666666667
kasus_beda True
